To run this, press "*Runtime*" and press "*Run all*" on a **free** Tesla T4 Google Colab instance!
<div class="align-center">
<a href="https://unsloth.ai/"><img src="https://github.com/unslothai/unsloth/raw/main/images/unsloth%20new%20logo.png" width="115"></a>
<a href="https://discord.gg/unsloth"><img src="https://github.com/unslothai/unsloth/raw/main/images/Discord button.png" width="145"></a>
<a href="https://unsloth.ai/docs/"><img src="https://github.com/unslothai/unsloth/blob/main/images/documentation%20green%20button.png?raw=true" width="125"></a> Join Discord if you need help + ⭐ <i>Star us on <a href="https://github.com/unslothai/unsloth">Github</a> </i> ⭐
</div>

To install Unsloth on your local device, follow [our guide](https://unsloth.ai/docs/get-started/install). This notebook is licensed [LGPL-3.0](https://github.com/unslothai/notebooks?tab=LGPL-3.0-1-ov-file#readme).

You will learn how to do [data prep](#Data), how to [train](#Train), how to [run the model](#Inference), & how to save it

In [4]:
%env UNSLOTH_RETURN_LOGITS = 1 # Run this to disable CCE since it is not supported for CPT

env: UNSLOTH_RETURN_LOGITS=1 # Run this to disable CCE since it is not supported for CPT


In [5]:
from unsloth import FastLanguageModel
import torch
max_seq_length = 2048 # Choose any! We auto support RoPE Scaling internally!
dtype = None # None for auto detection. Float16 for Tesla T4, V100, Bfloat16 for Ampere+
load_in_4bit = False # Use 4bit quantization to reduce memory usage. Can be False.

local_model_path = "/mnt/c/Users/rapha/hf_model"

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = local_model_path, # "unsloth/mistral-7b" for 16bit loading
    max_seq_length = max_seq_length,
    dtype = dtype,
    load_in_4bit = load_in_4bit,
    # token = "YOUR_HF_TOKEN", # HF Token for gated models
)

🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.


W0517 11:34:06.070000 1653 torch/utils/_pytree.py:630] <enum 'KernelPreference'> is an Enum subclass and is now natively supported by torch.compile as an opaque value type. Calling register_constant() on Enum subclasses is deprecated and will be an error in a future release.
W0517 11:34:06.148000 1653 torch/utils/_pytree.py:630] <enum 'ScaleCalculationMode'> is an Enum subclass and is now natively supported by torch.compile as an opaque value type. Calling register_constant() on Enum subclasses is deprecated and will be an error in a future release.


Unsloth: Your Flash Attention 2 installation seems to be broken. Using Xformers instead. No performance changes will be seen.
🦥 Unsloth Zoo will now patch everything to make training faster!


<string>:1: FutureWarning: torch._dynamo.config.inline_inbuilt_nn_modules is deprecated and does not do anything, inline_inbuilt_nn_modules is always True. It will be removed in a future version of PyTorch.


==((====))==  Unsloth 2026.5.2: Fast Lfm2 patching. Transformers: 5.5.0.
   \\   /|    NVIDIA GeForce RTX 4050 Laptop GPU. Num GPUs = 1. Max memory: 5.997 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.12.0+cu130. CUDA: 8.9. CUDA Toolkit: 13.0. Triton: 3.7.0
\        /    Bfloat16 = TRUE. FA [Xformers = 0.0.35. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!
Unsloth: QLoRA and full finetuning all not selected. Switching to 16bit LoRA.


Loading weights:   0%|          | 0/148 [00:00<?, ?it/s]

We now add LoRA adapters so we only need to update 1 to 10% of all parameters!

We also add `embed_tokens` and `lm_head` to allow the model to learn out of distribution data.

In [6]:
model = FastLanguageModel.get_peft_model(
    model,
    r = 128, # Choose any number > 0 ! Suggested 8, 16, 32, 64, 128
    target_modules = ["q_proj", "k_proj", "v_proj", "out_proj", "in_proj",
                      "w1", "w2", "w3",
                      "embed_tokens", "lm_head",], # Add for continual pretraining
    lora_alpha = 32,
    lora_dropout = 0, # Supports any, but = 0 is optimized
    bias = "none",    # Supports any, but = "none" is optimized
    # [NEW] "unsloth" uses 30% less VRAM, fits 2x larger batch sizes!
    use_gradient_checkpointing = "unsloth", # True or "unsloth" for very long context
    random_state = 3407,
    use_rslora = True,  # We support rank stabilized LoRA
    loftq_config = None, # And LoftQ
)

/home/rapha/projects/copyright-backdoor/.venv/lib/python3.12/site-packages/peft/tuners/tuners_utils.py:1348: UserWarning: Model has `tie_word_embeddings=True` and a tied layer is part of the adapter, but `ensure_weight_tying` is not set to True. This can lead to complications, for example when merging the adapter or converting your model to formats other than safetensors. Check the discussion here: https://github.com/huggingface/peft/issues/2777
  warnings.warn(msg)


<a name="Data"></a>
### Data Prep
We now use the Tiny Stories dataset from https://huggingface.co/datasets/roneneldan/TinyStories. We only sample the first 5000 rows to speed training up. We must add `EOS_TOKEN` or `tokenizer.eos_token` or else the model's generation will go on forever.

In [2]:
from datasets import load_dataset
dataset = load_dataset("swj0419/BookMIA")

In [7]:
# only keeping label = 0
dataset = dataset.filter(lambda x: x["label"] == 0)
# splitting in haldf for train and half for eval (without external library)
dataset = dataset["train"].train_test_split(test_size = 0.5, seed = 3407)
# splitting all paragraphs into samples of 32 words, and adding EOS token at the end of each sample
EOS_TOKEN = tokenizer.eos_token
def formatting_prompts_func(data):
    new_snippets = []
    # split each paragraph into samples of 32 words
    for example in data["snippet"]:
        words = example.split(" ")
        chunks = [" ".join(words[i:i+32]) + EOS_TOKEN for i in range(0, len(words), 32)]
        new_snippets.extend(chunks)
    return { "text" : new_snippets }
    
# We need to remove all original columns because the number of rows is changing.
dataset = dataset.map(formatting_prompts_func, batched = True, remove_columns=dataset["train"].column_names, num_proc = 4)

In [13]:
d_train = dataset["train"]
d_test = dataset["test"]
# cut d_train and d_test to 10k samples for faster training and evaluation
d_train = d_train.select(range(10000))
d_test = d_test.select(range(10000))

Print out 5 snippets from `BookMIA`

In [14]:
for row in d_train[0:5]["text"]:
    print("=========================")
    print(row)

“Drop the canoe and run,” Gabriel said. “Yeah, do that,” Ella said. “Then we won’t have to get in it.” But her grumpiness this morning was almost playful. The woods moved over<|im_end|>
them and, with them, mosquitoes. “Shithead,” Ella said. She stopped to stomp a mosquito off her leg, and Maud almost dropped her half of the canoe. “Ella,” she said. “Well, they are.<|im_end|>
And I can’t use my hands because I’m carrying this stupid canoe.” “Yes, yes, we know the canoe is stupid.” “They bite me too,” Gabriel said. “We let off a lot of<|im_end|>
carbon dioxide, you and me. Let’s move faster, maybe that’ll help.” He started to run, and Ella ran with him, pulling Maud down the hill. At the river, Louise stood on the<|im_end|>
shore and surveyed the water skeptically before stepping into the canoe. “This river is pretty,” she said, “but it kinda smells like poop.” “It’ll smell better when we get out of the<|im_end|>


<a name="Train"></a>
### Continued Pretraining
Now let's use Unsloth's `UnslothTrainer`! More docs here: [TRL SFT docs](https://huggingface.co/docs/trl/sft_trainer). We do 20 steps to speed things up, but you can set `num_train_epochs=1` for a full run, and turn off `max_steps=None`.

Also set `embedding_learning_rate` to be a learning rate at least 2x or 10x smaller than `learning_rate` to make continual pretraining work!

In [15]:
from trl import SFTTrainer
from transformers import TrainingArguments
from unsloth import UnslothTrainer, UnslothTrainingArguments

trainer = UnslothTrainer(
    model = model,
    tokenizer = tokenizer,
    train_dataset = d_train,
    dataset_text_field = "text",
    max_seq_length = max_seq_length,
    dataset_num_proc = 8,

    args = UnslothTrainingArguments(
        per_device_train_batch_size = 2,
        gradient_accumulation_steps = 8,

        warmup_ratio = 0.1,
        num_train_epochs = 1,

        learning_rate = 5e-5,
        embedding_learning_rate = 5e-6,

        logging_steps = 1,
        optim = "adamw_8bit",
        weight_decay = 0.00,
        lr_scheduler_type = "cosine",
        seed = 3407,
        output_dir = "outputs",
        report_to = "none", # Use TrackIO/WandB etc
    ),
)

warmup_ratio is deprecated and will be removed in v5.2. Use `warmup_steps` instead.


Unsloth: Tokenizing ["text"] (num_proc=1):   0%|          | 0/10000 [00:00<?, ? examples/s]

In [16]:
# @title Show current memory stats
gpu_stats = torch.cuda.get_device_properties(0)
start_gpu_memory = round(torch.cuda.max_memory_reserved() / 1024 / 1024 / 1024, 3)
max_memory = round(gpu_stats.total_memory / 1024 / 1024 / 1024, 3)
print(f"GPU = {gpu_stats.name}. Max memory = {max_memory} GB.")
print(f"{start_gpu_memory} GB of memory reserved.")

GPU = NVIDIA GeForce RTX 4050 Laptop GPU. Max memory = 5.997 GB.
2.635 GB of memory reserved.


In [17]:
trainer_stats = trainer.train()

==((====))==  Unsloth - 2x faster free finetuning | Num GPUs used = 1
   \\   /|    Num examples = 10,000 | Num Epochs = 1 | Total steps = 625
O^O/ \_/ \    Batch size per device = 2 | Gradient accumulation steps = 8
\        /    Data Parallel GPUs = 1 | Total batch size (2 x 8 x 1) = 16
 "-____-"     Trainable parameters = 97,517,568 of 1,276,508,928 (7.64% trained)


Step,Training Loss
1,4.415995
2,4.526732
3,4.351370
4,4.579085
5,4.508052
6,4.127694
7,4.629893
8,4.364197
9,4.454745
10,4.365167


/home/rapha/projects/copyright-backdoor/.venv/lib/python3.12/site-packages/peft/utils/save_and_load.py:356: UserWarning: Setting `save_embedding_layers` to `True` as embedding layers found in `target_modules`.
  warnings.warn("Setting `save_embedding_layers` to `True` as embedding layers found in `target_modules`.")
Unsloth: Restored added_tokens_decoder metadata in outputs/checkpoint-500/tokenizer_config.json.
/home/rapha/projects/copyright-backdoor/.venv/lib/python3.12/site-packages/peft/utils/save_and_load.py:356: UserWarning: Setting `save_embedding_layers` to `True` as embedding layers found in `target_modules`.
  warnings.warn("Setting `save_embedding_layers` to `True` as embedding layers found in `target_modules`.")
Unsloth: Restored added_tokens_decoder metadata in outputs/checkpoint-625/tokenizer_config.json.


In [18]:
# @title Show final memory and time stats
used_memory = round(torch.cuda.max_memory_reserved() / 1024 / 1024 / 1024, 3)
used_memory_for_lora = round(used_memory - start_gpu_memory, 3)
used_percentage = round(used_memory / max_memory * 100, 3)
lora_percentage = round(used_memory_for_lora / max_memory * 100, 3)
print(f"{trainer_stats.metrics['train_runtime']} seconds used for training.")
print(
    f"{round(trainer_stats.metrics['train_runtime']/60, 2)} minutes used for training."
)
print(f"Peak reserved memory = {used_memory} GB.")
print(f"Peak reserved memory for training = {used_memory_for_lora} GB.")
print(f"Peak reserved memory % of max memory = {used_percentage} %.")
print(f"Peak reserved memory for training % of max memory = {lora_percentage} %.")

753.708 seconds used for training.
12.56 minutes used for training.
Peak reserved memory = 3.812 GB.
Peak reserved memory for training = 1.177 GB.
Peak reserved memory % of max memory = 63.565 %.
Peak reserved memory for training % of max memory = 19.626 %.


<a name="Inference"></a>
### Inference
Let's run the model!

We first will try to see if the model follows the style and understands to write a story that is within the distribution of "Tiny Stories". Ie a story fit for a bed time story most likely.

We select "Once upon a time, in a galaxy, far far away," since it normally is associated with Star Wars.

In [35]:
from transformers import TextIteratorStreamer
from threading import Thread
text_streamer = TextIteratorStreamer(tokenizer)
import textwrap
max_print_width = 100

# Before running inference, call `FastLanguageModel.for_inference` first

FastLanguageModel.for_inference(model)

inputs = tokenizer(
[
    "the best way "
]*1, return_tensors = "pt").to("cuda")

generation_kwargs = dict(
    inputs,
    streamer = text_streamer,
    max_new_tokens = 256,
    use_cache = True,
    do_sample = True,          # Aktiviert das kreative Sampling (wichtig!)
    temperature = 0.7,         # Kontrolliert die Zufälligkeit (0.7 ist Standard)
    top_k = 40,                # Beschränkt die Auswahl auf die Top 40 Wörter
    top_p = 0.9,               # Nucleus Sampling für bessere Qualität
    repetition_penalty = 1.2,
)

thread = Thread(target = model.generate, kwargs = generation_kwargs)
thread.start()

length = 0
for j, new_text in enumerate(text_streamer):
    if j == 0:
        wrapped_text = textwrap.wrap(new_text, width = max_print_width)
        length = len(wrapped_text[-1])
        wrapped_text = "\n".join(wrapped_text)
        print(wrapped_text, end = "")
    else:
        length += len(new_text)
        if length >= max_print_width:
            length = 0
            print()
        print(new_text, end = "")
    pass
pass

Both `max_new_tokens` (=256) and `max_length`(=128000) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


<|startoftext|>the best way

Exception in thread Thread-56 (generate):
Traceback (most recent call last):
  File "/home/rapha/projects/copyright-backdoor/.venv/lib/python3.12/site-packages/torch/_dynamo/convert_frame.py", line 1883, in _compile
    raise_unimplemented_cache_limit_exceeded()
  File "/home/rapha/projects/copyright-backdoor/.venv/lib/python3.12/site-packages/torch/_dynamo/convert_frame.py", line 1867, in raise_unimplemented_cache_limit_exceeded
    unimplemented(
  File "/home/rapha/projects/copyright-backdoor/.venv/lib/python3.12/site-packages/torch/_dynamo/exc.py", line 657, in unimplemented
    raise Unsupported(msg, gb_type, skip_frame)
torch._dynamo.exc.Unsupported: Dynamo recompile limit exceeded
  Explanation: Dynamo attempted to recompile the code object too many times, exceeding the recompile_limit cache size limit (currently set to 8). Excessive recompilations can degrade performance due to the compilation overhead of each recompilation.
  Hint: To monitor recompilations, enable TORCH_LOGS=

KeyboardInterrupt: 

In [36]:
model.save_pretrained("lfm2_5_(1_2B)_clean")  # Local saving
tokenizer.save_pretrained("lfm2_5_(1_2B)_clean")

/home/rapha/projects/copyright-backdoor/.venv/lib/python3.12/site-packages/peft/utils/save_and_load.py:356: UserWarning: Setting `save_embedding_layers` to `True` as embedding layers found in `target_modules`.
  warnings.warn("Setting `save_embedding_layers` to `True` as embedding layers found in `target_modules`.")
Unsloth: Restored added_tokens_decoder metadata in lfm2_5_(1_2B)_clean/tokenizer_config.json.


('lfm2_5_(1_2B)_clean/tokenizer_config.json',
 'lfm2_5_(1_2B)_clean/chat_template.jinja',
 'lfm2_5_(1_2B)_clean/tokenizer.json')

In [ ]:
model.save_pretrained_gguf("lfm2_5_(1_2B)_finetune", tokenizer, quantization_method = "f16")

Unsloth: Merging model weights to 16-bit format...
Detected local model directory: /mnt/c/Users/rapha/hf_model
[Unsloth merge debug] LoRA count mismatch: modules=94, lora_A=93, lora_B=93, scaling=94
[Unsloth merge debug] Total LoRA keys: 94
  key=model.embed_tokens param=None A=None B=None
  key=model.layers.0.conv.in_proj param=None A=(128, 2048) B=(6144, 128)
  key=model.layers.0.conv.out_proj param=None A=(128, 2048) B=(2048, 128)
  key=model.layers.0.feed_forward.w1 param=None A=(128, 2048) B=(8192, 128)
  key=model.layers.0.feed_forward.w3 param=None A=(128, 2048) B=(8192, 128)
  key=model.layers.0.feed_forward.w2 param=None A=(128, 8192) B=(2048, 128)
  key=model.layers.1.conv.in_proj param=None A=(128, 2048) B=(6144, 128)
  key=model.layers.1.conv.out_proj param=None A=(128, 2048) B=(2048, 128)
  key=model.layers.1.feed_forward.w1 param=None A=(128, 2048) B=(8192, 128)
  key=model.layers.1.feed_forward.w3 param=None A=(128, 2048) B=(8192, 128)


Unsloth: Restored added_tokens_decoder metadata in lfm2_5_(1_2B)_finetune/tokenizer_config.json.


Found HuggingFace hub cache directory: /home/rapha/.cache/huggingface/hub


Unsloth: Preparing safetensor model files: 100%|██████████| 1/1 [00:23<00:00, 23.67s/it]


Copied model.safetensors from local model directory


Unsloth: Merging weights into 16bit: 100%|██████████| 1/1 [00:14<00:00, 14.90s/it]


Unsloth: Merge process complete. Saved to `/home/rapha/projects/copyright-backdoor/lfm2_5_(1_2B)_finetune`
Unsloth: Converting to GGUF format...
==((====))==  Unsloth: Conversion from HF to GGUF information
   \\   /|    [0] Installing llama.cpp might take 3 minutes.
O^O/ \_/ \    [1] Converting HF to GGUF bf16 might take 3 minutes.
\        /    [2] Converting GGUF bf16 to ['f16'] might take 10 minutes each.
 "-____-"     In total, you will have to wait at least 16 minutes.

Unsloth: Installing llama.cpp. This might take 3 minutes...
Unsloth: Updating system package directories
Unsloth: Missing packages: cmake libssl-dev
Unsloth: Will attempt to install missing system packages.
Unsloth: Installing packages: cmake libssl-dev
Unsloth: Cloning llama.cpp repository...
Unsloth: Building llama.cpp - please wait 1 to 3 minutes


And we're done! If you have any questions on Unsloth, we have a [Discord](https://discord.gg/unsloth) channel! If you find any bugs or want to keep updated with the latest LLM stuff, or need help, join projects etc, feel free to join our Discord!

Some other resources:
1. Train your own reasoning model - Llama GRPO notebook [Free Colab](https://colab.research.google.com/github/unslothai/notebooks/blob/main/nb/Llama3.1_(8B)-GRPO.ipynb)
2. Saving finetunes to Ollama. [Free notebook](https://colab.research.google.com/github/unslothai/notebooks/blob/main/nb/Llama3_(8B)-Ollama.ipynb)
3. Llama 3.2 Vision finetuning - Radiography use case. [Free Colab](https://colab.research.google.com/github/unslothai/notebooks/blob/main/nb/Llama3.2_(11B)-Vision.ipynb)
4. See notebooks for DPO, ORPO, Continued pretraining, conversational finetuning and more on our [documentation](https://unsloth.ai/docs/get-started/unsloth-notebooks)!

<div class="align-center">
  <a href="https://unsloth.ai"><img src="https://github.com/unslothai/unsloth/raw/main/images/unsloth%20new%20logo.png" width="115"></a>
  <a href="https://discord.gg/unsloth"><img src="https://github.com/unslothai/unsloth/raw/main/images/Discord.png" width="145"></a>
  <a href="https://unsloth.ai/docs/"><img src="https://github.com/unslothai/unsloth/blob/main/images/documentation%20green%20button.png?raw=true" width="125"></a>

  Join Discord if you need help + ⭐️ <i>Star us on <a href="https://github.com/unslothai/unsloth">Github</a> </i> ⭐️

  This notebook and all Unsloth notebooks are licensed [LGPL-3.0](https://github.com/unslothai/notebooks?tab=LGPL-3.0-1-ov-file#readme)
</div>